# Phase 2 — Data Cleaning & Feature Transformation Pipeline
**Project**: InsightPath — SAS CU Hackathon Round 2 Analytics
**Module**: `notebooks/02_data_cleaning/02_data_cleaning.ipynb`

### Analytical Objectives of Phase 2
This notebook executes the end-to-end data cleaning, standardization, feature derivation, and validation workflows across the four project datasets based strictly on empirical audit findings from Phase 1:
1. **Junior Data Scientist Skill Traits (`JDS`)**: Strip 32 trailing Excel blanks ($171 \rightarrow 139$), standardize `maths_stats_skills`, validate 1.0–5.0 scale, and construct Baseline ($N=139$) and Sensitivity ($N=137$, excluding ambiguous duplicate ID 3291) analytical datasets.
2. **Senior Data Scientist Personality Traits (`SDS`)**: Sanitize column headers (eliminate leading/adjacent whitespace to snake_case), validate raw psychometric traits on the 17–68 scale, retain all 161 records and duplicate IDs for observational clustering.
3. **Data Science Jobs**: Parse salary strings with 'L' suffix into continuous numeric values (Lakhs INR), verify strict mathematical ordering ($\text{min} \le \text{avg} \le \text{max}$), derive salary spread, and compute log-transformed hiring volume ($N=1,602$).
4. **Analytics Jobs**: Parse multi-format experience text into numeric intervals ($[\text{min}, \text{max}, \text{midpoint}]$), transform discrete salary brackets into ordinal ranks (1–6) and midpoints, derive top-50 multi-hot skill indicators, collapse 1,355 locations into 7 macro clusters, map 10,097 job titles into 6 standardized role families, and impute text missingness ($N=15,841$).
5. **Data Governance & Integrity**: Maintain complete raw data immutability verified via SHA-256 cryptographic hashing before and after all operations.


In [ ]:
import os
import sys
import hashlib
import pandas as pd
import numpy as np

# Set project root to path
sys.path.append(os.path.abspath("../.."))

from src.preprocessing.load_raw import verify_raw_integrity, load_raw_phase2
from src.preprocessing.transformation_log import TransformationLogger
from src.preprocessing.clean_jds import clean_jds_dataset
from src.preprocessing.clean_sds import clean_sds_dataset
from src.preprocessing.clean_data_science_jobs import clean_data_science_jobs_dataset
from src.preprocessing.clean_analytics_jobs import clean_analytics_jobs_dataset
from src.preprocessing.validate_processed_data import validate_processed_datasets

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 1000)
print("Environment and modules initialized successfully.")


## 1. Cryptographic Raw Data Integrity Verification (Pre-Execution)
Before performing any operations in memory, we verify that all source files in `data/raw/` match their root counterparts and exhibit zero prior modifications.


In [ ]:
df_pre_integrity = verify_raw_integrity(raw_dir="../../data/raw", project_root="../..")
df_pre_integrity[["filename", "size_bytes", "root_sha256", "status"]]


## 2. Ingestion & Transformation Logging Setup
We initialize the `TransformationLogger` to record all atomic data modifications, preserving full auditability.


In [ ]:
logger = TransformationLogger(output_path="../../outputs/tables/phase2_transformation_log.csv")
raw_datasets = load_raw_phase2(raw_dir="../../data/raw")

for name, df in raw_datasets.items():
    print(f"Loaded Raw Dataset: '{name}' | Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")


## 3. Junior Data Scientist Skill Traits (`JDS`) Cleaning
- Removes 32 trailing Excel spreadsheet rows ($171 \rightarrow 139$).
- Standardizes column `maths-stats_skills` $\rightarrow$ `maths_stats_skills`.
- Validates 5 skill domains within $[1.0, 5.0]$ and binary target in $\{0, 1\}$.
- Constructs JDS Baseline ($N=139$) and JDS Sensitivity ($N=137$, excluding conflicting duplicate ID 3291).


In [ ]:
df_jds_interim, df_jds_baseline, df_jds_sensitivity = clean_jds_dataset(
    raw_datasets["JDS Skill Traits"],
    logger=logger
)

print(f"JDS Raw Rows: {len(raw_datasets['JDS Skill Traits'])}")
print(f"JDS Cleaned Rows: {len(df_jds_baseline)} (Removed exactly 32 blank trailing rows)")
print(f"JDS Sensitivity Rows: {len(df_jds_sensitivity)} (Excluded duplicate ID 3291)")
print("\nJDS Baseline Summary Statistics:")
df_jds_baseline.describe().round(2)


## 4. Senior Data Scientist Personality Traits (`SDS`) Cleaning
- Sanitizes headers (` extraversion` $\rightarrow$ `extraversion`, `success_ classification_ high_low` $\rightarrow$ `success_classification_high_low`).
- Validates 5 Big Five psychometric scores on the raw 17–68 scale (zero clipping or scaling).
- Validates binary success target in $\{0, 1\}$.
- Retains all 161 rows and duplicate ID pairs for observational analysis.


In [ ]:
df_sds_interim, df_sds_processed = clean_sds_dataset(
    raw_datasets["SDS Personality Traits"],
    logger=logger
)

print(f"SDS Cleaned Rows: {len(df_sds_processed)}")
print(f"SDS Sanitized Columns: {list(df_sds_processed.columns)}")
print("\nSDS Summary Statistics (Raw Psychometric Scale 17-68):")
df_sds_processed.describe().round(2)


## 5. Data Science Jobs Cleaning & Feature Engineering
- Trims whitespace from company names without entity distortion.
- Parses salary strings (`'10L'` $\rightarrow$ `10.0`).
- Validates strict mathematical ordering: $\text{min\_salary\_lakh} \le \text{avg\_salary\_lakh} \le \text{max\_salary\_lakh}$.
- Derives `salary_spread_lakh` and relative `salary_spread_ratio`.
- Computes `log10_num_of_jobs` ($N=1,602$).


In [ ]:
df_ds_interim, df_ds_processed = clean_data_science_jobs_dataset(
    raw_datasets["DataScience Jobs"],
    logger=logger
)

print(f"Data Science Jobs Processed Shape: {df_ds_processed.shape}")
print(f"Salary Ordering Violations: {((df_ds_processed['min_salary_lakh'] > df_ds_processed['avg_salary_lakh']) | (df_ds_processed['avg_salary_lakh'] > df_ds_processed['max_salary_lakh'])).sum()}")
print("\nDerived Numeric Salary & Volume Summary:")
df_ds_processed[["min_salary_lakh", "avg_salary_lakh", "max_salary_lakh", "salary_spread_lakh", "num_of_jobs", "log10_num_of_jobs"]].describe().round(2)


## 6. Analytics Jobs Cleaning, Feature Engineering & Harmonization
- Parses experience text into `min_experience`, `max_experience`, `midpoint_experience`.
- Maps 6 salary brackets to `salary_rank` (1–6), `salary_midpoint` (Lakhs), and `is_high_salary` (binary $\ge 15\text{L}$).
- Generates Top-50 multi-hot skill indicator features from empirical posting frequency profile.
- Aggregates 1,355 raw locations into 7 macro tech clusters.
- Classifies 10,097 raw job titles into 6 standardized role families.
- Imputes text missingness in `job_type`, `job_description`, and `key_skills`.


In [ ]:
df_aj_interim, df_aj_processed = clean_analytics_jobs_dataset(
    raw_datasets["Analytics Jobs"],
    logger=logger,
    top_k_skills=50,
    skill_output_path="../../outputs/tables/skill_frequency_profile.csv"
)

print(f"Analytics Jobs Processed Shape: {df_aj_processed.shape}")
print(f"Location Clusters: {df_aj_processed['location_cluster'].value_counts().to_dict()}")
print(f"Role Families: {df_aj_processed['job_role_family'].value_counts().to_dict()}")
print(f"Salary Brackets: {df_aj_processed['salary_rank'].value_counts().sort_index().to_dict()}")


## 7. Post-Cleaning Quality Validation Suite
We evaluate all datasets across 13 structural and quality checks.


In [ ]:
interim_dict = {
    "jds_interim": df_jds_interim,
    "sds_interim": df_sds_interim,
    "data_science_jobs_interim": df_ds_interim,
    "analytics_jobs_interim": df_aj_interim
}
processed_dict = {
    "jds_processed": df_jds_baseline,
    "jds_sensitivity_3291_removed": df_jds_sensitivity,
    "sds_processed": df_sds_processed,
    "data_science_jobs_processed": df_ds_processed,
    "analytics_jobs_processed": df_aj_processed
}

df_val_summary, df_before_after = validate_processed_datasets(
    raw_datasets=raw_datasets,
    interim_datasets=interim_dict,
    processed_datasets=processed_dict,
    output_dir="../../outputs/tables"
)

print("=== Post-Cleaning Validation Summary ===")
display(df_val_summary)

print("\n=== Selected Before vs After Metrics ===")
display(df_before_after.head(15))


## 8. Cryptographic Raw Data Immutability Re-Verification (Post-Execution)
We re-verify that all raw source files remain 100% byte-for-byte identical after all pipeline operations.


In [ ]:
df_post_integrity = verify_raw_integrity(raw_dir="../../data/raw", project_root="../..")
print("=== Post-Execution Raw File Cryptographic Verification ===")
display(df_post_integrity[["filename", "size_bytes", "root_sha256", "raw_sha256", "is_identical", "status"]])
assert df_post_integrity["is_identical"].all(), "FATAL: Raw files were mutated during execution!"
